In [3]:
# ── Setup Session 4 : Structured Streaming ─────────────────────────────
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ.get("PATH", "")

In [4]:
from pathlib import Path
import time
import datetime

# ── Chemins ─────────────────────────────────────────────────────────────────
DATA_DIR           = Path("data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"

for chemin in [VELIB_CONSOLIDE]:
    assert chemin.exists(), f"Fichier manquant : {chemin} -- relancez Session 2"

for chemin in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    chemin.mkdir(parents=True, exist_ok=True)

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Streaming"
SHUFFLE_PARTS = 8

print("Chemins configurés.")
print(f"  Stream source : {STREAM_SOURCE_DIR}")
print(f"  Checkpoint    : {STREAM_CHECKPOINT}")
print(f"  Delta alertes : {DELTA_ALERTES}")


Chemins configurés.
  Stream source : data/output/stream_input
  Checkpoint    : data/output/checkpoints
  Delta alertes : data/output/delta/alertes


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, TimestampType,
)
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "8g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
    # Streaming : délai minimal entre deux micro-batchs
    .config("spark.sql.streaming.minBatchesToRetain", "2")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake activé")
print(f"Spark UI : http://localhost:4040")


:: loading settings :: url = jar:file:/Users/ms/Library/CloudStorage/SynologyDrive-cefedemaura/cours/hetic/spark_project/venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ms/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ms/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ea8ce3e5-38da-47ac-a255-b258fdc2e0b1;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 69ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.0 from central in [default]
	io.delta#delta-storage;4.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|               

Spark 4.0.4 -- Delta Lake activé
Spark UI : http://localhost:4040


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50839)
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/ms/Library/CloudStorage/SynologyDrive-

---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


In [6]:
# Vérification : le simulateur tourne-t-il ?
import time, os

def compter_fichiers_stream(attente_sec: int = 5) -> int:
    """Attend et compte les fichiers JSON produits par le simulateur."""
    time.sleep(attente_sec)
    fichiers = list(STREAM_SOURCE_DIR.glob("*.json"))
    return len(fichiers)

nb_fichiers = compter_fichiers_stream(3)
if nb_fichiers == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print(f"  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
    print(f"\n  Commande à lancer dans un terminal :")
    print(f"  python scripts/simulateur_flux.py --output {STREAM_SOURCE_DIR} --vitesse 3")
else:
    print(f"[OK] {nb_fichiers} fichier(s) JSON détecté(s) dans {STREAM_SOURCE_DIR}")
    dernier_fichier = sorted(STREAM_SOURCE_DIR.glob("*.json"))[-1]
    print(f"  Dernier fichier : {dernier_fichier.name}")
    with open(dernier_fichier, encoding="utf-8") as f:
        premiere_ligne = f.readline()
    print(f"  Aperçu : {premiere_ligne[:200]}")


[OK] 129 fichier(s) JSON détecté(s) dans data/output/stream_input
  Dernier fichier : batch_000128.json
  Aperçu : {"station_id": 753437, "nom_station": "Benjamin Godard - Victor Hugo", "code_arr": 16, "capacite": 35, "velos_meca": 3, "velos_elec": 2, "bornettes_libres": 7, "horodatage": "2020-11-27T23:06:00Z"}



---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [7]:
# Schéma du flux JSON produit par le simulateur
schema_flux = StructType([
    StructField("station_id",       IntegerType(),   False),
    StructField("nom_station",      StringType(),    True),
    StructField("code_arr",         IntegerType(),   True),
    StructField("capacite",         IntegerType(),   True),
    StructField("velos_meca",       IntegerType(),   True),
    StructField("velos_elec",       IntegerType(),   True),
    StructField("bornettes_libres", IntegerType(),   True),
    StructField("horodatage",       TimestampType(), False),
])

# Création du DataFrame de streaming
# maxFilesPerTrigger : au plus 2 fichiers traités par micro-batch
stream_df = (
    spark.readStream
    .format("json")
    .schema(schema_flux)
    .option("maxFilesPerTrigger", 2)
    .load(str(STREAM_SOURCE_DIR))
)

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame

Est un streaming DataFrame : True
Colonnes : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage']


In [9]:
# Ajout des colonnes calculées -- exactement comme en batch
stream_enrichi = (
    stream_df
    .withColumn("velos_total",
                F.col("velos_meca") + F.col("velos_elec"))
    .withColumn("taux_occupation",
                F.round(
                    (F.col("velos_meca") + F.col("velos_elec")) / F.col("capacite"),
                    3
                ))
    .withColumn("est_vide",
                (F.col("velos_meca") + F.col("velos_elec")) == 0)
)
print("Colonnes après enrichissement :", stream_enrichi.columns)

Colonnes après enrichissement : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage', 'velos_total', 'taux_occupation', 'est_vide']


---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [10]:
# Requête de streaming vers la console
# outputMode :
#   "append"  -- écrit uniquement les nouvelles lignes (défaut pour les flux sans agrégation)
#   "complete" -- réécrit l'intégralité du résultat à chaque batch
#   "update"  -- écrit uniquement les lignes qui ont changé

q_console = (
    stream_enrichi
    .select("station_id", "nom_station", "code_arr",
            "horodatage", "velos_total", "taux_occupation", "est_vide")
    .writeStream
    .outputMode("append")
    .format("console")
    .option("numRows", 10)
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .queryName("q_console_debug")
    .start()
)

# La requête tourne en arrière-plan.
time.sleep(20)
print(f"Statut de la requête : {q_console.status}")
print(f"Batchs traités : {q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}")
q_console.stop()
print("Requête console arrêtée.")

26/09/21 17:22:54 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/4b/kxqwhtfx2_j8wy3m7r55_lzr0000gn/T/temporary-309a863b-5ccc-4e63-a267-131e5dc1065f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/09/21 17:22:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----------+-----------------------------------+--------+-------------------+-----------+---------------+--------+
|station_id|nom_station                        |code_arr|horodatage         |velos_total|taux_occupation|est_vide|
+----------+-----------------------------------+--------+-------------------+-----------+---------------+--------+
|753437    |Benjamin Godard - Victor Hugo      |16      |2020-11-26 14:06:00|4          |0.114          |false   |
|246819    |André Mazet - Saint-André des Arts |6       |2020-11-26 14:06:00|23         |0.418          |false   |
|623455    |Charonne - Robert et Sonia Delauney|NULL    |2020-11-26 14:06:00|0          |0.0            |true    |
|503963    |Toudouze - Clauzel                 |9       |2020-11-26 14:06:00|0          |0.0            |true    |
|176815    |Mairie du 12ème                    |12      |2020-11-26 14:06:00|3          |0.1      

26/09/21 17:23:14 WARN DAGScheduler: Failed to cancel job group 754006dc-e786-4084-8700-7716efb83b59. Cannot find active jobs for it.
26/09/21 17:23:14 WARN DAGScheduler: Failed to cancel job group 754006dc-e786-4084-8700-7716efb83b59. Cannot find active jobs for it.


---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```


In [11]:
from pyspark.sql.functions import window

# Agrégation glissante : disponibilité moyenne par arrondissement
# Fenêtre de 10 minutes, avançant toutes les 2 minutes
# Le watermark de 5 min permet à Spark de fermer les fenêtres et d'émettre en mode append
df_fenetre_arr = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(
        F.col("code_arr"),
        window(F.col("horodatage"), "10 minutes", "2 minutes").alias("fenetre"),
    )
    .agg(
        F.round(F.avg("taux_occupation"), 3).alias("taux_moyen"),
        F.sum("velos_total").alias("velos_total_sum"),
        F.count("*").alias("nb_snapshots"),
    )
    .withColumn("fenetre_debut", F.col("fenetre.start"))
    .withColumn("fenetre_fin",   F.col("fenetre.end"))
    .drop("fenetre")
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()

Schéma de la requête fenêtrée :
root
 |-- code_arr: integer (nullable = true)
 |-- taux_moyen: double (nullable = true)
 |-- velos_total_sum: long (nullable = true)
 |-- nb_snapshots: long (nullable = false)
 |-- fenetre_debut: timestamp (nullable = true)
 |-- fenetre_fin: timestamp (nullable = true)



In [12]:
# Écriture vers Delta Lake (sink delta) -- mode append
# En mode append avec watermark, Spark n'émet chaque fenêtre
# qu'une seule fois, après que le watermark l'a "fermée".
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

q_fenetres = (
    df_fenetre_arr
    .writeStream
    .outputMode("append")
    .format("delta")
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "fenetres"))
    .option("path", path_fenetres)
    .trigger(processingTime="10 seconds")
    .queryName("q_fenetres_arrondissement")
    .start()
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print("En attente de 3 batchs...")
time.sleep(35)
print(f"Dernier progrès : {q_fenetres.lastProgress}")

26/09/21 17:24:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Requête démarrée : q_fenetres_arrondissement
ID               : 0bb3ef4c-3ed4-4f08-88bd-202962359653
En attente de 3 batchs...


26/09/21 17:24:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Dernier progrès : {
    "id": "0bb3ef4c-3ed4-4f08-88bd-202962359653",
    "runId": "2359b125-27d7-4ef8-b34f-09ec26a64aa8",
    "name": "q_fenetres_arrondissement",
    "timestamp": "2026-09-21T15:24:40.004Z",
    "batchId": 4,
    "batchDuration": 980,
    "numInputRows": 2782,
    "inputRowsPerSecond": 278.17218278172186,
    "processedRowsPerSecond": 2838.775510204082,
    "durationMs": {
        "addBatch": 836,
        "commitOffsets": 31,
        "getBatch": 4,
        "latestOffset": 60,
        "queryPlanning": 10,
        "triggerExecution": 980,
        "walCommit": 36
    },
    "eventTime": {
        "avg": "2020-11-26T15:15:30.000Z",
        "max": "2020-11-26T15:25:00.000Z",
        "min": "2020-11-26T15:06:00.000Z",
        "watermark": "2020-11-26T14:42:00.000Z"
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 645,
            "numRowsUpdated": 430,
            "allUpdatesTimeMs": 105,
            "numRows

In [13]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    df_fenetres_result = spark.read.format("delta").load(path_fenetres)
    df_fenetres_result.orderBy(F.desc("fenetre_debut")).show(30, truncate=False)
except Exception as erreur:
    print(f"Pas encore de données : {erreur}")
    print("Attendez encore quelques secondes et relancez cette cellule.")

+--------+----------+---------------+------------+-------------------+-------------------+
|code_arr|taux_moyen|velos_total_sum|nb_snapshots|fenetre_debut      |fenetre_fin        |
+--------+----------+---------------+------------+-------------------+-------------------+
|8       |0.354     |579            |49          |2020-11-26 15:32:00|2020-11-26 15:42:00|
|28      |0.1       |3              |1           |2020-11-26 15:32:00|2020-11-26 15:42:00|
|24      |0.249     |42             |6           |2020-11-26 15:32:00|2020-11-26 15:42:00|
|13      |0.264     |662            |61          |2020-11-26 15:32:00|2020-11-26 15:42:00|
|3       |0.523     |207            |14          |2020-11-26 15:32:00|2020-11-26 15:42:00|
|92      |0.084     |20             |7           |2020-11-26 15:32:00|2020-11-26 15:42:00|
|22      |0.161     |137            |33          |2020-11-26 15:32:00|2020-11-26 15:42:00|
|9       |0.268     |332            |45          |2020-11-26 15:32:00|2020-11-26 15:42:00|

---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [14]:
# Table d'alertes : schéma
schema_alertes = StructType([
    StructField("station_id",    IntegerType(),   False),
    StructField("nom_station",   StringType(),    True),
    StructField("code_arr",      IntegerType(),   True),
    StructField("debut_rupture", TimestampType(), True),
    StructField("fin_rupture",   TimestampType(), True),
    StructField("duree_min",     IntegerType(),   True),
    StructField("ts_alerte",     TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide)
df_alertes_vide = spark.createDataFrame([], schema_alertes)
(
    df_alertes_vide.write
    .format("delta")
    .mode("overwrite")
    .save(str(DELTA_ALERTES))
)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")

Table d'alertes initialisée : data/output/delta/alertes


In [16]:
from delta.tables import DeltaTable

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
# clé = station_id, valeur = {"nom_station", "code_arr", "debut": Timestamp}
etat_ruptures: dict[int, dict] = {}

def traiter_batch_alertes(batch_df, batch_id: int) -> None:
    """Appelée par foreachBatch pour chaque micro-batch.

    Détecte les transitions vide/non-vide et enregistre les alertes
    dans la table Delta quand la rupture dépasse 10 minutes.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    if batch_df.rdd.isEmpty():
        return

    # Agrégation au niveau station sur ce batch :
    # horodatage min/max, nb d'obs vides, nb total, arrondissement, nom
    etat_batch = (
        batch_df
        .groupBy("station_id", "nom_station", "code_arr")
        .agg(
            F.min("horodatage").alias("ts_min"),
            F.max("horodatage").alias("ts_max"),
            F.sum(F.col("est_vide").cast("int")).alias("nb_vide"),
            F.count("*").alias("nb_total"),
        )
    ).collect()

    alertes = []

    for ligne in etat_batch:
        station_id = ligne["station_id"]
        est_vide_maintenant = (ligne["nb_vide"] == ligne["nb_total"])

        if est_vide_maintenant and station_id not in etat_ruptures:
            # Transition non-vide → vide : début de rupture
            etat_ruptures[station_id] = {
                "nom_station": ligne["nom_station"],
                "code_arr":    ligne["code_arr"],
                "debut":       ligne["ts_min"],
            }

        elif not est_vide_maintenant and station_id in etat_ruptures:
            # Transition vide → non-vide : fin de rupture
            debut   = etat_ruptures[station_id]["debut"]
            fin     = ligne["ts_max"]
            duree_min = int((fin.timestamp() - debut.timestamp()) / 60)

            if duree_min >= 10:
                alertes.append({
                    "station_id":    station_id,
                    "nom_station":   etat_ruptures[station_id]["nom_station"],
                    "code_arr":      etat_ruptures[station_id]["code_arr"],
                    "debut_rupture": debut,
                    "fin_rupture":   fin,
                    "duree_min":     duree_min,
                    "ts_alerte":     datetime.datetime.utcnow(),
                })
            del etat_ruptures[station_id]

    if alertes:
        df_nouvelles_alertes = spark.createDataFrame(alertes, schema_alertes)
        (
            df_nouvelles_alertes.write
            .format("delta")
            .mode("append")
            .save(str(DELTA_ALERTES))
        )
        print(f"[Batch {batch_id}] {len(alertes)} alerte(s) enregistrée(s)")

In [17]:
# Lancement de la requête d'alerte
# - withWatermark : tolérance de 5 minutes sur l'horodatage
# - foreachBatch  : appel de traiter_batch_alertes à chaque micro-batch
# - checkpoint    : sauvegarde de l'état Spark (pas de l'état Python etat_ruptures)
# - trigger       : 10 secondes entre deux batchs
q_alertes = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .writeStream
    .outputMode("append")
    .foreachBatch(traiter_batch_alertes)
    .option("checkpointLocation", str(STREAM_CHECKPOINT / "alertes"))
    .trigger(processingTime="10 seconds")
    .queryName("q_alertes_rupture")
    .start()
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(60)   # laisser tourner 1 minute pour accumuler des événements

# Lecture des alertes produites
df_alertes_result = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes_result.count()}")
df_alertes_result.orderBy(F.desc("ts_alerte")).show(20, truncate=False)


26/09/21 17:27:20 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Requête d'alertes démarrée. En attente de données...
[Batch 1] 49 alerte(s) enregistrée(s)
[Batch 2] 68 alerte(s) enregistrée(s)
[Batch 3] 28 alerte(s) enregistrée(s)
[Batch 4] 64 alerte(s) enregistrée(s)
[Batch 5] 36 alerte(s) enregistrée(s)
[Batch 6] 112 alerte(s) enregistrée(s)

Alertes enregistrées : 357
+----------+--------------------------------------+--------+-------------------+-------------------+---------+--------------------------+
|station_id|nom_station                           |code_arr|debut_rupture      |fin_rupture        |duree_min|ts_alerte                 |
+----------+--------------------------------------+--------+-------------------+-------------------+---------+--------------------------+
|687736    |Quai de Valmy                         |NULL    |2020-11-26 15:32:00|2020-11-26 17:41:00|129      |2026-09-21 15:28:20.252745|
|775509    |Guy Môquet - Compoint                 |17      |2020-11-26 16:34:00|2020-11-26 17:41:00|67       |2026-09-21 15:28:20.252721|


---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [18]:
import json as json_module

def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20) -> None:
    """Écrit un fichier JSON avec des horodatages retardés dans le passé.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.
    """
    ts_maintenant = datetime.datetime.utcnow()
    ts_retarde    = ts_maintenant - datetime.timedelta(minutes=retard_minutes)
    horodatage_str = ts_retarde.strftime("%Y-%m-%dT%H:%M:%SZ")

    fichier_sortie = STREAM_SOURCE_DIR / f"late_{retard_minutes}min_{int(ts_maintenant.timestamp())}.json"

    # Stations fictives pour illustrer les données tardives
    stations_test = [
        {"station_id": 100001, "nom_station": "Station-Test-Retard", "code_arr": 1,
         "capacite": 20, "velos_meca": 0, "velos_elec": 0, "bornettes_libres": 20},
        {"station_id": 100002, "nom_station": "Station-Test-Retard-2", "code_arr": 2,
         "capacite": 15, "velos_meca": 5, "velos_elec": 3, "bornettes_libres": 7},
    ]

    with open(fichier_sortie, "w", encoding="utf-8") as fout:
        for idx in range(nb_lignes):
            station = stations_test[idx % len(stations_test)]
            enregistrement = {**station, "horodatage": horodatage_str}
            fout.write(json_module.dumps(enregistrement) + "\n")

    message = (f"[{datetime.datetime.now().strftime('%H:%M:%S')}] "
               f"Données tardives injectées : retard={retard_minutes}min, "
               f"fichier={fichier_sortie.name}, horodatage={horodatage_str}")
    print(message)
    fichier_log = OUTPUT_DIR / "simulateur.log"
    with open(fichier_log, "a", encoding="utf-8") as flog:
        flog.write(message + "\n")


# Scénario 1 : retard 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(15)

# Scénario 2 : retard 12 min -- hors watermark -> données ignorées par les agrégations
injecter_donnees_tardives(retard_minutes=12, nb_lignes=10)
time.sleep(15)

print("\nStatut des requêtes actives :")
for requete in spark.streams.active:
    prog = requete.lastProgress
    print(f"  {requete.name:<35} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}")

[17:28:21] Données tardives injectées : retard=3min, fichier=late_3min_1789997301.json, horodatage=2026-09-21T15:25:21Z
[17:28:36] Données tardives injectées : retard=12min, fichier=late_12min_1789997316.json, horodatage=2026-09-21T15:16:36Z

Statut des requêtes actives :
  q_fenetres_arrondissement           -- inputRows : 2782
  q_alertes_rupture                   -- inputRows : 4173


---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [19]:
# Arrêt de toutes les requêtes actives
requetes_actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[r.name for r in requetes_actives]}")

for requete in requetes_actives:
    requete.stop()
    print(f"  Arrêtée : {requete.name}")

fichier_log = OUTPUT_DIR / "simulateur.log"
with open(fichier_log, "a", encoding="utf-8") as flog:
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    flog.write(f"[{ts}] Toutes les requêtes de streaming arrêtées\n")

print("\nToutes les requêtes de streaming sont arrêtées.")

Requêtes actives à arrêter : ['q_fenetres_arrondissement', 'q_alertes_rupture']
  Arrêtée : q_fenetres_arrondissement
  Arrêtée : q_alertes_rupture

Toutes les requêtes de streaming sont arrêtées.


26/09/21 17:29:13 WARN DAGScheduler: Failed to cancel job group 2359b125-27d7-4ef8-b34f-09ec26a64aa8. Cannot find active jobs for it.
26/09/21 17:29:13 WARN DAGScheduler: Failed to cancel job group 2359b125-27d7-4ef8-b34f-09ec26a64aa8. Cannot find active jobs for it.
26/09/21 17:29:13 WARN DAGScheduler: Failed to cancel job group 7fe2839e-c8c4-4a7f-a577-2c892a7b3f50. Cannot find active jobs for it.
26/09/21 17:29:13 WARN DAGScheduler: Failed to cancel job group 7fe2839e-c8c4-4a7f-a577-2c892a7b3f50. Cannot find active jobs for it.


In [20]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")

=== Résultats finaux ===

-- Fenêtres arrondissement --
  12642 fenêtres enregistrées
+--------+----------+---------------+------------+-------------------+-------------------+
|code_arr|taux_moyen|velos_total_sum|nb_snapshots|fenetre_debut      |fenetre_fin        |
+--------+----------+---------------+------------+-------------------+-------------------+
|28      |0.367     |11             |1           |2020-11-27 06:16:00|2020-11-27 06:26:00|
|19      |0.125     |195            |57          |2020-11-27 06:16:00|2020-11-27 06:26:00|
|18      |0.154     |222            |51          |2020-11-27 06:16:00|2020-11-27 06:26:00|
|2       |0.252     |153            |22          |2020-11-27 06:16:00|2020-11-27 06:26:00|
|27      |0.222     |28             |5           |2020-11-27 06:16:00|2020-11-27 06:26:00|
|14      |0.134     |231            |54          |2020-11-27 06:16:00|2020-11-27 06:26:00|
|51      |0.231     |26             |3           |2020-11-27 06:16:00|2020-11-27 06:26:00|
|42 

---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [ ]:
spark.stop()
print("SparkSession arrêtée. À demain !")
